# 02 - Loss Function Comparison
Compare CE, CE+Focal, ArcFace, and CosFace with fixed backbone and training pipeline.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
import src.feature_cache as feature_cache
import src.wandb_utils as wandb_utils
from src.models import ArcFaceHeadModel, CEHeadModel, CosFaceHeadModel, FocalLoss, build_backbone
from src.training import fit_embeddings
from src.utils import get_device, set_seed

load_dotenv(project_root / ".env")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = get_device()
device

## Base Config


In [ ]:
config = {
    # Paths
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("../checkpoints/e2_loss_functions"),
    "submission_dir": Path("../submissions/e2_loss_functions"),
    "embeddings_cache_dir": Path("../checkpoints/embedding_cache"),

    # Model
    "backbone_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
    "input_size": 448,
    "embedding_dim": 256,
    "hidden_dim": 512,
    "dropout": 0.3,
    "arcface_margin": 0.5,
    "cosface_margin": 0.35,
    "arcface_scale": 64.0,
    
    # Training
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 100,
    "patience": 10,
    "val_split": 0.2,
    "num_workers": 2,
    "seed": RANDOM_SEED,
    "focal_gamma": 2.0,
}

config["checkpoint_dir"].mkdir(exist_ok=True)


## Load Data


In [ ]:
def load_data(config, backbone):
    train_df = data.load_train_df(config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    train_data, val_data = data.train_val_split(
        train_df,
        val_split=config["val_split"],
        seed=config["seed"],
        stratify_col="ground_truth",
    )

    backbone_train_loader, backbone_val_loader = data.create_backbone_dataloaders(
        backbone,
        train_data,
        val_data,
        img_dir=config["data_dir"] / "train" / "train",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
    )

    pairs_df = data.load_test_pairs_df(config["data_dir"])
    unique_images = sorted(set(pairs_df["query_image"].unique()) | set(pairs_df["gallery_image"].unique()))
    test_df = pd.DataFrame({"filename": unique_images})

    test_loader = data.create_test_loader(
        backbone,
        test_df,
        img_dir=config["data_dir"] / "test" / "test",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
    )

    return {
        "label_encoder": label_encoder,
        "num_classes": num_classes,
        "backbone_train_loader": backbone_train_loader,
        "backbone_val_loader": backbone_val_loader,
        "pairs_df": pairs_df,
        "test_loader": test_loader,
    }

## Create Embeddings with fixed Backbone


In [ ]:
backbone = build_backbone(config["backbone_name"], pretrained=True).to(device)
backbone.eval()
backbone_out_dim = getattr(backbone, "num_features", None)
if backbone_out_dim is None:
    raise ValueError("Backbone output dimension not found")

data_bundle = load_data(config, backbone)
label_encoder = data_bundle["label_encoder"]
num_classes = data_bundle["num_classes"]
pairs_df = data_bundle["pairs_df"]
test_loader = data_bundle["test_loader"]

cache_dir = config["embeddings_cache_dir"]
cache_dir.mkdir(exist_ok=True)
cache_key = f"{config['backbone_name'].replace(':', '_').replace('/', '_')}_{config['input_size']}"

train_cache = cache_dir / f"train_{cache_key}.npz"
val_cache = cache_dir / f"val_{cache_key}.npz"

train_embeddings, train_labels = feature_cache.get_or_create_embeddings(
    train_cache,
    backbone,
    data_bundle["backbone_train_loader"],
    device,
)
val_embeddings, val_labels = feature_cache.get_or_create_embeddings(
    val_cache,
    backbone,
    data_bundle["backbone_val_loader"],
    device,
)

param_count_backbone = sum(p.numel() for p in backbone.parameters())
print(f"Backbone params: {param_count_backbone:,}")
print(f"Train embeddings shape: {train_embeddings.shape} | Val embeddings shape: {val_embeddings.shape}")


## Run Loss Comparison


In [ ]:
experiments = [
    {"name": "CrossEntropy", "loss_name": "ce"},
    {"name": "CrossEntropyFocal", "loss_name": "ce_focal"},
    {"name": "ArcFace", "loss_name": "arcface"},
    {"name": "CosFace", "loss_name": "cosface"},
]

def build_head_model(loss_name, input_dim, num_classes, config):
    common_kwargs = dict(
        input_dim=input_dim,
        num_classes=num_classes,
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"],
    )

    if loss_name == "arcface":
        return ArcFaceHeadModel(
            **common_kwargs,
            margin=config["arcface_margin"],
            scale=config["arcface_scale"],
        )

    if loss_name == "ce":
        return CEHeadModel(
            **common_kwargs,
        )

    if loss_name == "ce_focal":
        return CEHeadModel(
            **common_kwargs,
        )

    if loss_name == "cosface":
        return CosFaceHeadModel(
            **common_kwargs,
            margin=config["cosface_margin"],
            scale=config["arcface_scale"],
        )

    raise ValueError(f"Unknown loss_name: {loss_name}")


In [ ]:
from src.wandb_utils import log_submission_artifact


def run_experiment(experiment, run_idx, total_runs):
    run_name = f"02-{experiment['name']}"
    loss_name = experiment["loss_name"]

    print("=" * 80)
    print(f"Run {run_idx}/{total_runs}: {run_name}")

    train_loader, val_loader = data.create_embedding_dataloaders(
        train_embeddings,
        train_labels,
        val_embeddings,
        val_labels,
        batch_size=config["batch_size"],
    )

    head_model = build_head_model(loss_name, backbone_out_dim, num_classes, config).to(device)
    param_count = sum(p.numel() for p in head_model.parameters())

    run_config = dict(config)
    run_config["loss_name"] = loss_name
    wandb = wandb_utils.init_wandb(
        run_config,
        run_name=run_name,
        param_count=param_count,
        param_count_backbone=param_count_backbone,
    )

    if loss_name == "ce_focal":
        criterion = FocalLoss(gamma=config["focal_gamma"])
    else:
        criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        [p for p in head_model.parameters() if p.requires_grad],
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )

    checkpoint_name = f"{run_name}.pth"
    checkpoint_path = config["checkpoint_dir"] / checkpoint_name
    results = fit_embeddings(
        model=head_model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        label_encoder=label_encoder,
        wandb_run=wandb,
        checkpoint_name=checkpoint_name,
    )

    wandb.run.summary["best_val_mAP"] = results["best_map"]
    wandb.run.summary["best_val_loss"] = results["best_val_loss"]
    wandb.run.summary["best_epoch"] = results["best_epoch"]
    wandb.run.summary["total_epochs"] = results["epochs_ran"]

    submission_path = config["submission_dir"] / f"submission_{run_name}.csv"
    inference.create_submission_backbone(
        backbone,
        head_model,
        device,
        pairs_df,
        test_loader,
        submission_path,
    )

    wandb_utils.log_checkpoint_artifact(
        wandb,
        checkpoint_path,
        artifact_name=run_name,
        description="checkpoint",
    )

    log_submission_artifact(
        wandb,
        submission_path,
        artifact_name=run_name,
        description="Competition submission file",
    )

    wandb.finish()


for run_idx, experiment in enumerate(experiments, start=1):
    run_experiment(experiment, run_idx, len(experiments))
